# PARAMO · 24 barras — hidrología y transmisión

Lee la misma capa numérica que el visor público 1.0. Conserva por separado **medias de conjuntos**, **realizaciones seleccionadas** y **parámetros de entrada**. El horizonte es 2025–2050, con doce meses de 720 h por año modelado. El modelo reducido de 6 barras y sus pruebas UC se mantienen independientes.

Ejecute las celdas en orden. El notebook funciona con los datos incluidos; la importación opcional de tablas por central se habilita al final. Ninguna celda llama al optimizador.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'assets/js/dashboard_data.js').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Abra el notebook desde la carpeta que contiene index.html y assets.')

def read_js_json(relative):
    text = (ROOT / relative).read_text(encoding='utf-8')
    return json.loads(text.split('=', 1)[1].strip().removesuffix(';'))

DATA = read_js_json('assets/js/dashboard_data.js')
DETAIL = read_js_json('assets/js/national_detail.js')
PLANT_DATA = read_js_json('assets/js/reservoir_plants.js')
print('Visor público: datos locales cargados. Modelo seleccionado: EC24.')

## 1. Cobertura disponible

In [ ]:
storage = pd.DataFrame(DATA['national']['reservoirRep'])
transmission = pd.DataFrame(DATA['national']['transmissionStates'])
generation = pd.DataFrame(DATA['national']['monthlyGen'])
summary = pd.DataFrame(DATA['national']['summary'])
plant_inputs = pd.DataFrame(DETAIL['plants'])
display(pd.DataFrame([
    {'Capa': 'Almacenamiento agregado', 'Registros': len(storage), 'Tipo': 'Realizaciones identificadas · mensual'},
    {'Capa': 'Estados de transmisión', 'Registros': len(transmission), 'Tipo': 'Media ponderada por corredor y año'},
    {'Capa': 'Generación mensual', 'Registros': len(generation), 'Tipo': 'Realizaciones identificadas · tecnología'},
    {'Capa': 'Embalses representados', 'Registros': len(plant_inputs), 'Tipo': 'Parámetros de entrada, Tabla 4.10'},
    {'Capa': 'Curvas por central importadas', 'Registros': len(PLANT_DATA['means']), 'Tipo': 'Opcional; fuente original requerida'},
]))

## 2. Selección de análisis

In [ ]:
SCENARIO = 'BRIDGE'  # REF, BRIDGE, NZT
YEAR = 2050          # 2025–2050
MONTH = 1            # 1–12
assert SCENARIO in ('REF', 'BRIDGE', 'NZT')
assert 2025 <= YEAR <= 2050 and 1 <= MONTH <= 12
COLORS = {'Baseline': '#168b87', 'Adverse': '#c43e7d'}
selected_storage = storage.query('Scenario == @SCENARIO and Year == @YEAR').sort_values(['Hydrology', 'Month'])
display(selected_storage[['Hydrology', 'Representative_w', 'Year', 'Month', 'StorageHm3', 'TurbineHm3', 'SpillHm3']])

## 3. Ciclos mensuales de almacenamiento

Cada línea conserva el identificador de la realización correspondiente. La comparación describe las dos configuraciones: también cambian el calendario estacional y los costos de candidatos nucleares.

In [ ]:
fig = go.Figure()
for hydro, rows in selected_storage.groupby('Hydrology', sort=False):
    run = rows['Representative_w'].iloc[0]
    fig.add_scatter(x=rows.Month, y=rows.StorageHm3, mode='lines+markers',
                    name=f'{hydro} · {run}', line={'color': COLORS[hydro], 'width': 3})
fig.update_layout(template='plotly_white', title=f'{SCENARIO} · {YEAR} · almacenamiento agregado',
                  xaxis_title='Mes modelado [720 h]', yaxis_title='Volumen al cierre del mes [hm³]',
                  legend={'orientation': 'h'}, height=450)
fig.show()

## 4. Recarga y vaciado dentro del año

La primera diferencia se calcula **dentro de cada año**. Enero permanece sin valor de variación porque el volumen inicial anual no está en la tabla agregada. Diciembre del año anterior no se utiliza como sustituto.

In [ ]:
delta = selected_storage.copy()
delta['DeltaStorage_hm3'] = delta.groupby(['Hydrology', 'Scenario', 'Year'], sort=False)['StorageHm3'].diff()
fig = go.Figure()
for hydro, rows in delta.groupby('Hydrology', sort=False):
    fig.add_bar(x=rows.Month, y=rows.DeltaStorage_hm3, name=hydro, marker_color=COLORS[hydro])
fig.update_layout(template='plotly_white', barmode='group', title=f'{SCENARIO} · {YEAR} · cambio de almacenamiento',
                  xaxis_title='Mes modelado [720 h]', yaxis_title='ΔV [hm³]', height=430)
fig.show()
assert delta.query('Month == 1')['DeltaStorage_hm3'].isna().all()

## 5. Matriz de almacenamiento: meses y años independientes

In [ ]:
HYDROLOGY = 'Baseline'
rows = storage.query('Scenario == @SCENARIO and Hydrology == @HYDROLOGY')
matrix = rows.pivot(index='Year', columns='Month', values='StorageHm3').sort_index()
fig = go.Figure(go.Heatmap(x=matrix.columns, y=matrix.index, z=matrix.values,
                         colorscale='Blues', zmin=0, zmax=659.33,
                         colorbar={'title': 'hm³'},
                         hovertemplate='Año %{y}<br>Mes %{x}<br>Volumen %{z:,.2f} hm³<extra></extra>'))
fig.update_layout(template='plotly_white', title=f'{SCENARIO} · {HYDROLOGY} · años independientes',
                  xaxis_title='Mes modelado [720 h]', yaxis_title='Año modelado', height=550)
fig.update_yaxes(autorange='reversed')
fig.show()

## 6. Operaciones hidráulicas y energía

El volumen turbinado y el vertimiento son **sumas sobre plantas**. El agua reutilizada en la cascada aparece en más de una operación; esta suma no es el caudal externo de la cuenca.

In [ ]:
water = selected_storage.query('Hydrology == @HYDROLOGY')
fig = go.Figure()
fig.add_bar(x=water.Month, y=water.TurbineHm3, name='Turbinado · suma de plantas', marker_color='#168b87')
fig.add_bar(x=water.Month, y=water.SpillHm3, name='Vertimiento / bypass · suma de plantas', marker_color='#e3a62f')
fig.update_layout(template='plotly_white', barmode='stack', title=f'{SCENARIO} · {HYDROLOGY} · {YEAR}',
                  xaxis_title='Mes modelado [720 h]', yaxis_title='Operaciones hidráulicas [hm³/mes modelado]', height=420)
fig.show()

## 7. Medias de almacenamiento

Las barras representan las medias del conjunto de estados mensuales durante 2025–2050. Las trayectorias seleccionadas se presentan por separado.


In [ ]:
means = pd.DataFrame(DETAIL['meanStorage'])
means['MeanStorage_hm3'] = pd.to_numeric(means['MeanStorage_hm3'])
display(means.pivot(index='Scenario', columns='Campaign', values='MeanStorage_hm3'))
display(plant_inputs[['name','cascade','status','capacity_MW','vmin_hm3','vmax_hm3','qmax_hm3']])

## 8. Estados de inversión en transmisión

Reconductorizar un corredor se resume como participación de realizaciones. Los circuitos nuevos estándar y reconductorizados se suman para obtener un **número esperado de circuitos**, que puede superar uno. La visualización geográfica interactiva se encuentra en la pestaña **Transmission & Reconductoring** del dashboard.

In [ ]:
tx = transmission.query('Scenario == @SCENARIO and Year == @YEAR').copy()
assert len(tx) == 62  # 31 corredores × dos casos hidrológicos
assert ((tx.ExpectedNewStandard + tx.ExpectedNewReconductored - tx.ExpectedNewCircuits).abs() < 1e-8).all()
display(tx.groupby('Hydrology')[['ExpectedNewCircuits','ReconductoringShare']].sum().rename(
    columns={'ReconductoringShare':'ExpectedReconductoredCorridors'}))
display(tx[['Hydrology','FromNode','ToNode','ReconductoringShare','ExpectedNewStandard','ExpectedNewReconductored','ExpectedNewCircuits']])

In [ ]:
history = transmission.query('Scenario == @SCENARIO').groupby(['Hydrology','Year'], as_index=False).ExpectedNewCircuits.sum()
fig = go.Figure()
for hydro, rows in history.groupby('Hydrology', sort=False):
    fig.add_scatter(x=rows.Year, y=rows.ExpectedNewCircuits, mode='lines+markers', name=hydro,
                    line={'color':COLORS[hydro], 'shape':'hv'})
fig.update_layout(template='plotly_white', title=f'{SCENARIO} · circuitos nuevos en servicio',
                  xaxis_title='Año', yaxis_title='Número esperado de circuitos [–]', height=430)
fig.show()

## 9. Control de la capa recibida

Estos controles comprueban claves, cobertura y sumas de la capa de visualización. No vuelven a certificar el problema de optimización ni completan fuentes ausentes.

In [ ]:
checks = {
    '1872 registros de volumen agregado': len(storage) == 1872,
    '4836 estados de corredor': len(transmission) == 4836,
    'claves de almacenamiento únicas': not storage.duplicated(['Hydrology','Scenario','Year','Month']).any(),
    'claves de transmisión únicas': not transmission.duplicated(['Hydrology','Scenario','Year','FromNode','ToNode']).any(),
    'participaciones de reconductorización entre 0 y 1': transmission.ReconductoringShare.between(-1e-9, 1+1e-9).all(),
    'circuitos nuevos no negativos': (transmission.ExpectedNewCircuits >= -1e-9).all(),
    'cinco plantas del caso 24N': len(plant_inputs) == 5,
}
display(pd.DataFrame([{'Control':k, 'Resultado':bool(v)} for k,v in checks.items()]))
assert all(checks.values())

## 10. Importación opcional de las tablas mensuales por central

El paquete research dataset recibido marca como pendientes `reservoir_storage_statistics_by_plant.csv` y `reservoir_representative_by_plant.csv`. Si estas tablas existen en su compilación local, indique la carpeta de `10_canonical` a continuación. El importador verifica **SHA-256, cobertura, unidades y balances** contra el registro de research dataset. El valor predeterminado mantiene la capa pública intacta.

Cada tabla debe tener 9.360 filas. El importador conserva las realizaciones de la fuente —incluido w90 si corresponde— en vez de reemplazarlas por los identificadores de la tabla agregada.

In [ ]:
SOURCE_DIR = None  # Ejemplo: Path(r'C:\ruta\al\build\10_canonical')
WRITE_BROWSER_LAYER = False
sys.path.insert(0, str(ROOT/'scripts'))
from import_reservoir_tables import collect, publish

if SOURCE_DIR is None:
    print('Se conserva la capa agregada incluida. No se han importado datos por central.')
else:
    imported = collect(Path(SOURCE_DIR))
    display(pd.DataFrame(imported['provenance']))
    if WRITE_BROWSER_LAYER:
        publish(imported)
        print('Capa de embalses actualizada. Recargue el dashboard.')
    else:
        print('Validación completada, sin escritura. Active WRITE_BROWSER_LAYER para publicar la capa local.')

## Publicación

El repositorio público recibe **el contenido de la raíz del ZIP**, con `index.html` junto a `assets`, `data` y `docs`. El notebook es opcional para visitantes: los gráficos funcionan con HTML/JavaScript estáticos y Plotly incluido localmente. Consulte `LEEME_PUBLICAR_GITHUB.md`.

La importación anterior modifica únicamente `assets/js/reservoir_plants.js`. Los resultados del modelo y las fuentes originales se conservan.